In [3]:
import torch
import torchvision as tv
import os
import pathlib

In [4]:
if torch.cuda.is_available():
    device = 'cuda' 
elif torch.backends.mps.is_available():
    device = 'mps' # if on mac
else:
    device = 'cpu' # if mps not available
print(f'using device {device}')

using device mps


In [9]:
! python --version || { echo "[ERROR] Python not found"; exit 1; }
  

Python 3.11.11


In [10]:
! python -u ./DPItorch/DPI_PET.py \
  --activity_path   ./dataset/brain64_tumor_activity_map/brain64_tumor_FDG_K1_40min.mat \
  --sinogram_path   ./dataset/brain64_tumor_sinogram/brain64_tumor_FDG_K1_40min.mat \
  --gmat_path       ./dataset/G_system_matrix_64.mat \
  --ri_path         ./dataset/brain64_tumor_sinogram/brain64_tumor_FDG_K1_40min_ri.mat \
  --ytrue_path      ./dataset/brain64_tumor_sinogram/brain64_tumor_FDG_K1_40min_ytrue.mat \
  --save_dir      ./checkpoint/glow64 \
  --model_form     glow \
  --image_size     64  \
  --n_epoch        10 \
  --n_flow         16 \
  --n_batch        64 \
  --tv 5 || { echo "[ERROR] Python script execution failed"; exit 1; }

Torch version: 2.6.0
Device selected: cpu
CUDA not available, using CPU -- slower training
---------- 1. 读取 PET 数据 (activity & sinogram) ----------
✔ activity shape: (64, 64, 60, 18)
✔ sinogram shape: (64, 160, 60, 18)
====== Using image_size: 64x64, frame: 10, roi: 30 ======
✔ 观测数据 y_vec (noisy sinogram) shape: torch.Size([10240])
---------- 2. 系统矩阵 G_sparse & 新增的 ci, ri, ytrue ----------
[INFO] 使用64x64数据集路径和G_system_matrix_64.mat
[系统矩阵] 尝试从以下路径加载: dataset/G_system_matrix_64.mat
✔ G shape: (10240, 4096) | non-zeros: 1391482 (3.32% non zeros)
✔ 系统矩阵存储在CPU上
✔ torch sparse A: torch.Size([10240, 4096]) | dtype: torch.float32 | device: cpu
✔ 背景噪声 ri_vec shape: torch.Size([10240])
✔ 无噪声真实投影 ytrue_vec shape: torch.Size([10240])
[验证第一步] ytrue与(A@act_gt)的MSE: 2.1364e-09
[验证第二步] ytrue与完整模型((A@act_gt)+ri)的MSE: 2.7628e+03
[验证第三步] y与(ytrue+ri)的MSE: 3.2192e+02
---------- 3. 构造 / 加载 Flow 生成器 ----------
/Users/linyuxuan/workSpace/DeepMed-Imaging-Reconstruction/DPIPET/DPItorch/generative_model/glow_mo